# Weak-lensing galaxy shape catalogue validation

## Maps

Contents.
- Creat convergence maps

> **_NOTE:_** Before running this notebook, set kernel to `main_set.ipynb'

In [1]:
from scipy.ndimage.filters import gaussian_filter
#from scipy.ndimage import gaussian_filter

In [2]:
from sp_validation.survey import *
from sp_validation.util import *
from sp_validation.basic import *
from sp_validation.plots import *
from sp_validation.cosmology import *

Could not import clmm, continuing...
Could not import clmm.modeling, continuing...
Could not import pyccl


In [3]:
sp_base = f"{os.environ['HOME']}/astro/repositories/github/sp_validation"

# The following commands will be replaced by import instructions, once the sp_validation scripts are stable
for sc in ['cosmology']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script

Could not import clmm, continuing...
Could not import clmm.modeling, continuing...
Could not import pyccl


# Pixelise ellipticities

In [1]:
# Compute number of pixels
Nx = int(size_x_deg / pixel_size_emap_amin * 60)
Ny = int(size_y_deg / pixel_size_emap_amin * 60)
print_stats(f'Numbers of elipticity pixels for KS93 = ({Nx}, {Ny})', stats_file, verbose=verbose)

NameError: name 'size_x_deg' is not defined

In [ ]:
# Bin in 2D
g1_tmp, g2_tmp = bin2d(
    x,
    y,
    npix=(Nx, Ny), 
    v=(g_calib_mc_ngmix[0], g_calib_mc_ngmix[1]),
    extent=(min_x, max_x, min_y, max_y)
)

g_calib_mc_ngmix_map = np.array([g1_tmp, g2_tmp])

# Create convergence maps

In [ ]:
# Transform gamma -> kappa using the Kaiser-Squires (1993) algorithm
kappaE, kappaB = ks93(g1_sign * g_calib_mc_ngmix_map[0], g2_sign * g_calib_mc_ngmix_map[1])

In [ ]:
# Smooth with Gaussian filter
kappaE_sm = gaussian_filter(kappaE, smoothing_scale_pix)
kappaB_sm = gaussian_filter(kappaB, smoothing_scale_pix)

# Get known cluster positions

In [ ]:
# Get cluster information

cluster_cat_name = 'HFI_PCCS_SZ-union_R2.08.fits.gz'
vos_dir = 'vos:cfis/cosmostat/cosmology/external/Planck'

clusters = get_clusters(cluster_cat_name, vos_dir, name)

print_stats(f"{len(clusters['ra'])} clusters found in {name} footprint", stats_file, verbose=verbose)

In [ ]:
clusters = {}
clusters['ra'] = [211.9] * 5
clusters['dec'] = [57.2, 55.9, 54.6, 53.3, 52]
clusters['ra'].append(ra_ngmix_mean)
clusters['dec'].append(dec_ngmix_mean)

In [ ]:
# Project cluster positions
x_cluster, y_cluster =  radec2xy(ra_ngmix_mean, dec_ngmix_mean, clusters['ra'], clusters['dec'])
clusters['x'] = x_cluster
clusters['y'] = y_cluster
print(clusters)

# Plot maps

In [ ]:
def get_ticks(loc, N, new_min, new_max):
    """Get ticks
    
    Return formatted axis ticks for plots.
    
    Parameters
    ----------
    loc : array of floats
        original tick locations
    N : number of pixels (in origina coordinates)
    new_min : float
        new coordinate minimum
    new_max : float
        new coordinate maximum
        
    Returns
    -------
    loc_new : array of floats
        new tick locations
    labels_new : array of strings
        new tick labels
    """
    
    loc_new = []
    labels_new = []

    for i in range(1, len(loc)-1):
        lab = loc[i] / N * (new_max - new_min) + new_min
        #print(loc[i], lab)
        loc_new.append(loc[i])
        labels_new.append(f'{lab:.1f}')

    return loc_new, labels_new


def plot_map(m, ra, dec, title, out_path, vlim=None, clusters=None):
    """Plot Map
    
    Plots 2D map.
    
    Parameters
    ----------
    m : 2D array of float
        map
    ra, dec : array of float
        coordinates, for axis ticks
    title : string
        plot title
    out_path : string
        output file path
    vlim : array(2) of float, optional, default=None
        limits of map values, if not given compute from map
    clusters :
        dictionary of cluster information, optional, default=None
    """
    
    plt.figure(figsize=(20, 20))

    # Plot image
    plt.imshow(m)
    #plt.grid()

    # Set colorbar
    if not vlim:
       vlim = plt.gci().get_clim()
    else:
        plt.gci().set_clim(vlim)
    plt.colorbar()

    # Transform axis labels
    ra_min, ra_max = ra_ngmix.min(), ra_ngmix.max()
    dec_min, dec_max = dec_ngmix.min(), dec_ngmix.max()

    loc, labels = plt.xticks()
    loc_ra, labels_ra = get_ticks(loc, Nx, ra_min, ra_max)
    plt.xticks(loc_ra, labels=labels_ra)
 
    loc, labels = plt.yticks()
    loc_dec, labels_dec = get_ticks(loc, Ny, dec_min, dec_max)
    plt.yticks(loc_dec, labels=labels_dec)
    
    # grid
    for x in loc_ra

    plt.gca().invert_yaxis()
    plt.gca().invert_xaxis()
    plt.xlabel('R.A. [deg]')
    plt.ylabel('Dec [deg]')

    mean_x = (min_x + max_x) / 2
    mean_y = (min_y + max_y) / 2
    clusters['x'][0] = 0 
    clusters['y'][0] = 0
    if clusters:
        x_cluster = (clusters['x'] + mean_x - min_x) / (max_x - min_x) * Nx
        y_cluster = (clusters['y'] + mean_y - min_y) / (max_y - min_y) * Ny
        dy = 0.02
        #x_cluster[0] = Nx/2
        #y_cluster[0] = Ny/2
        plt.plot(x_cluster, y_cluster, 'k+')
        
        plt.plot(Nx/2, Ny/2, 'rD', markersize=3)
    #for i in range(len(sdss_cluster_cut['z'])):
    #    plt.text(x_cluster[i], y_cluster[i] + dy * (plt.ylim()[0] - plt.ylim()[1]), round(sdss_cluster_cut['z'][i],3), color='k', fontsize=10, ha='center', va='center')
    #plt.text(x_cluster[i], y_cluster[i] - dy * (plt.ylim()[0] - plt.ylim()[1]), round(sdss_cluster_cut['M'][i]/1e14, 2), color='k', fontsize=10, ha='center', va='center')

    print(plt.ylim())
    
    plt.title(title)

    plt.savefig(out_path)
    
    return vlim

In [ ]:
print(min_x, min_y)
print(radec2xy(ra_ngmix_mean, dec_ngmix_mean, [ra_ngmix.min()], [dec_ngmix.min()]))
xx, yy = radec2xy(ra_ngmix_mean, dec_ngmix_mean, ra_ngmix, dec_ngmix)
print(xx.min(), yy.min())

In [ ]:
title = '$\kappa_{\\rm E}$'
out_path = f'{plot_dir}/kappa_E.png'

vlim = plot_map(kappaE_sm, ra_ngmix, dec_ngmix, title, out_path, clusters=clusters)

print(ra_ngmix.min(), ra_ngmix.max())
print(dec_ngmix.min(), dec_ngmix.max())
#for ra, dec in zip(clusters['ra'], clusters['dec']):
#    print(f'{ra:.2f} {dec:.2f}')

In [ ]:
title = '$\kappa_{\\rm B}$'
out_path = f'{plot_dir}/kappa_B.png'

#plot_map(kappaB_sm, ra_ngmix, dec_ngmix, title, out_path, vlim=vlim)

In [ ]:
help(radec2xy)